# 2. LoRA и вопросы к изображению
Задача: ответить yes/no на разные вопросы о конкретной фотографии. Название вида не
подаём во вход. Атрибуты вида не подставляем вместо атрибутов изображения.

Сначала собственный модуль и математика на CPU, затем processor, маска loss и LoRA
на Qwen3.5-0.8B. Для GPU запуска используйте одну карту: CUDA_VISIBLE_DEVICES=0.

In [ ]:
from pathlib import Path
import os, sys
# Start in repository root or its notebooks/solutions directory.
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "solutions"}:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("BIRD_DATA", str(ROOT / "data/cub8")))
assert (DATA / "manifest.jsonl").exists(), "Run scripts/prepare_data.py first"
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## TODO 1 — LoRALinear
$y=W_0x+b+(\alpha/r)BAx$, где $A\in R^{r\times d_{in}}$, $B\in R^{d_{out}\times r}$.
Заморозьте base, инициализируйте A случайно, B нулями. Напишите forward и merged().

До запуска предскажите: какой градиент на первом шаге равен нулю? Почему обе матрицы
нельзя занулить? Сколько параметров при d_in=d_out=1024 и r=8?

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=4):
        super().__init__()
        if rank < 1:
            raise ValueError("rank must be positive")
        self.base = base.requires_grad_(False)
        self.scale = alpha / rank
        self.A = nn.Parameter(base.weight.new_empty(rank, base.in_features))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.normal_(self.A, std=0.02)

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

    def merged(self):
        layer = nn.Linear(self.base.in_features, self.base.out_features,
                          bias=self.base.bias is not None).to(self.base.weight)
        with torch.no_grad():
            layer.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
            if layer.bias is not None:
                layer.bias.copy_(self.base.bias)
        return layer

In [ ]:
base = nn.Linear(5, 3)
adapter = LoRALinear(base, rank=2)
x = torch.randn(4, 5)
torch.testing.assert_close(adapter(x), base(x))
adapter(x).square().sum().backward()
assert adapter.A.grad.abs().sum() == 0
assert adapter.B.grad.abs().sum() > 0
assert base.weight.grad is None
with torch.no_grad():
    adapter.B.add_(.1)
torch.testing.assert_close(adapter(x), adapter.merged()(x))

In [ ]:
from birdlab.data import make_qa
from birdlab.vlm import load_model, QACollator, evaluate, messages
train_qa = make_qa(DATA, "train", balance=True)
val_qa = make_qa(DATA, "val", heldout_wording=True)
print(len(train_qa), len(val_qa), train_qa[0])
from transformers import set_seed
set_seed(42)
model, processor = load_model(target_mode=os.environ.get("BIRD_TARGETS", "all-linear"))

## Processor и токенизация
Рассмотрите IDs, токены и обратное декодирование. Совпадает ли число токенов с числом слов?
Посмотрите `pixel_values` и `image_grid_thw`. Почему картинку не обрабатывает tokenizer?

In [ ]:
for text in ["yes", "no", "house sparrow", "домовый воробей"]:
    ids = processor.tokenizer.encode(text, add_special_tokens=False)
    print(text, ids, processor.tokenizer.convert_ids_to_tokens(ids))
example = train_qa[0]
print(processor.apply_chat_template(messages(example, True), tokenize=False, enable_thinking=False))

## TODO 2 — маска loss
$L=-\sum_t m_t\log p(y_t|I,q,y_{<t})/\sum_t m_t$.
Верните копию input_ids; prompt и padding замените на -100. EOS ответа сохраняется.
Используйте attention_mask, а не равенство pad_token_id: pad и EOS могут совпадать.
Collator проверит границы prompt на настоящем chat template; не задавайте длину вручную.

In [ ]:
def answer_labels(input_ids, attention_mask, prompt_lengths):
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100
    for i, length in enumerate(prompt_lengths):
        labels[i, :length] = -100
    if (labels != -100).sum(dim=1).min().item() == 0:
        raise ValueError("No supervised answer tokens")
    return labels

In [ ]:
ids = torch.tensor([[1, 2, 7, 9, 9], [1, 2, 3, 8, 9]])
attention = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 1, 1, 1]])
assert answer_labels(ids, attention, [2, 3]).tolist() == [[-100,-100,7,9,-100],[-100,-100,-100,8,9]]
collator = QACollator(processor, label_function=answer_labels)
batch = collator(train_qa[:2])
for key, value in batch.items():
    print(key, tuple(value.shape))
for labels in batch["labels"]:
    print("Loss on:", processor.decode(labels[labels != -100]))
loss = model(**{k:v.to(DEVICE) for k,v in batch.items()}).loss
loss.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for n,p in model.named_parameters() if "lora_B" in n)
model.zero_grad(set_to_none=True)
del batch, loss

## До и после LoRA
Используем новые формулировки вопросов на validation. Сначала исходная модель (нулевые
LoRA-обновления), потом 50 шагов обучения. Ограниченный evaluation — только быстрый пилот.
Для окончательной оценки используйте все вопросы и отдельный test.
По умолчанию адаптируем линейные слои языковой части: этот вариант дал прирост в эталоне.
Для сравнения задайте BIRD_TARGETS=attention. Vision encoder и lm_head заморожены.

In [ ]:
import random
random.Random(42).shuffle(val_qa)
evaluation = val_qa[:int(os.environ.get("BIRD_EVAL_SIZE", "24"))]
before = evaluate(model, processor, evaluation)
print(before["tasks"])
from transformers import Trainer, TrainingArguments
RUN = ROOT / "runs/notebook_vlm"
args = TrainingArguments(output_dir=str(RUN), max_steps=int(os.environ.get("BIRD_STEPS", "50")), learning_rate=2e-4,
    per_device_train_batch_size=4, gradient_accumulation_steps=2,
    remove_unused_columns=False, report_to="none", save_strategy="no", logging_steps=5,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=False)
model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train_qa, data_collator=collator)
trainer.train()
model.config.use_cache = True
after = evaluate(model, processor, evaluation)
print(after["tasks"])
model.save_pretrained(RUN / "adapter")
processor.save_pretrained(RUN / "adapter")

## Эксперименты и выводы
1. Для каждого вопроса сравните с большинством ответов **в train**, а не test.
2. Перемешайте изображения: падает ли balanced accuracy? Учтите, что часть перемешанных
   фото может иметь тот же правильный ответ.
3. Сравните r=2 и r=8 при одинаковом числе шагов и данных.
4. Почему уменьшение teacher-forced loss не гарантирует правильную генерацию?
5. Каких вопросов модель не умеет решать? Не путайте новые формулировки с новыми задачами.
6. Бонус: добавьте вопросы о видимости частей из parts/part_locs.txt; не выводите
   видимость из отрицательной метки цвета.

In [ ]:
from birdlab.vlm import predict, MODEL_ID
from transformers import Qwen3_5ForConditionalGeneration
from peft import PeftModel
# Save a reference before releasing GPU memory; reload exactly the saved adapter.
reference = predict(model, processor, evaluation[0])
dtype = next(model.parameters()).dtype
del trainer, model
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
base = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="sdpa").to(DEVICE)
restored = PeftModel.from_pretrained(base, RUN / "adapter")
assert predict(restored, processor, evaluation[0]) == reference
print("Adapter reload OK:", reference)